# 03 — Modeling & Results (Demo)

End-to-end demo notebook used for the project defense:

1. Load dataset
2. Extract features
3. Train Random Forest **and** XGBoost with 5-fold CV
4. Evaluate on held-out test split
5. Display confusion matrices, ROC curves, and feature importances inline

Run top-to-bottom; every cell should succeed on either the synthetic `sample.csv` or a real CIC dataset.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

from src.features import extract_features
from src.preprocess import load_dataset, split

sns.set_theme(style="whitegrid")

## 1. Load data and build feature matrix

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "sample.csv"
data, y = load_dataset(DATA_PATH)
X = extract_features(data["query"]) if "query" in data.columns else data
print(f"rows: {len(X):,}   features: {X.shape[1]}")
X_train, X_test, y_train, y_test = split(X, y)

## 2. Cross-validated training

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    print(f"\n[{name}] 5-fold cross-validation:")
    row = {}
    for metric in ("accuracy", "f1", "roc_auc"):
        s = cross_val_score(model, X_train, y_train, cv=cv, scoring=metric, n_jobs=-1)
        row[metric] = float(s.mean())
        print(f"  {metric:10s}: {row[metric]:.4f}  (+/- {s.std():.4f})")
    cv_results[name] = row
pd.DataFrame(cv_results).T

## 3. Fit on the training split, evaluate on the held-out test split

In [ ]:
test_metrics = {}
roc_data = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    test_metrics[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, test_metrics[name]["roc_auc"])
    print(f"\n[{name}]")
    print(classification_report(y_test, y_pred, target_names=["benign", "tunneling"]))

pd.DataFrame(test_metrics).T.round(4)

## 4. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["benign", "tunneling"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 5. ROC curve comparison

In [ ]:
plt.figure(figsize=(5, 4.5))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC — model comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 6. Feature importance

Which features did each model rely on most? This is the headline plot for the presentation — it tells you *why* the detector works.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, model) in zip(axes, models.items()):
    importances = model.feature_importances_
    order = np.argsort(importances)[::-1]
    sns.barplot(
        x=importances[order],
        y=np.array(X.columns)[order],
        ax=ax,
        palette="viridis",
    )
    ax.set_title(f"{name} — feature importance")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 7. Save artefacts

In [ ]:
models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
joblib.dump(models["Random Forest"], models_dir / "rf.pkl")
joblib.dump(models["XGBoost"], models_dir / "xgb.pkl")
print(f"models written to {models_dir}")

## Conclusion

- Both classifiers separate benign from tunneling traffic with very high accuracy.
- Top features are length, entropy, and digit ratio — exactly what intuition predicts.
- Limitations: encrypted DoH bodies, low-and-slow tunneling, and queries crafted to mimic benign distributions are harder. See `reports/report.md` for the discussion.